# SVD 離線版：圖片 → 短片（OpenVINO，CPU 可跑）

本 notebook 是 [openvino_notebooks/stable-video-diffusion](https://github.com/openvinotoolkit/openvino_notebooks/tree/latest/notebooks/stable-video-diffusion)（Apache-2.0）的**離線特化版**：
官方版負責「轉檔＋推理」，這個版本**只做推理**——載入已經轉好的 OpenVINO IR，輸出 mp4。
（轉檔請在家、有網路的機器用 `tools/convert_svd.py` 跑一次。）

**前置**：
1. 已有 `models/svd-ov/`（含 `model/`、`scheduler/`、`feature_extractor/`）
2. 已安裝 `requirements-svd.txt` 的套件（torch、diffusers、transformers…）

預設路徑：在 kit 根目錄啟動 jupyter（`jupyter lab notebooks/video_svd_offline.ipynb`）。


## 0. 環境檢查

如果缺模型，先在（有網路的）機器執行：

```bash
pip install -r requirements-svd.txt
python tools/convert_svd.py --dest models/svd-ov
```


In [ ]:
import sys, time, subprocess, tempfile, shutil
from pathlib import Path
import numpy as np
import torch
import openvino as ov

# 找 kit 根目錄（往上找有 tools/ 的那層）
ROOT = Path.cwd().resolve()
while not (ROOT / "tools").exists() and ROOT.parent != ROOT:
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT / "tools"))

from lcm_scheduler import AnimateLCMSVDStochasticIterativeScheduler
from ov_stable_video_diffusion_helper import OVStableVideoDiffusionPipeline
from transformers import CLIPImageProcessor
from diffusers.utils import load_image

MODEL_ROOT = ROOT / "models" / "svd-ov"
print("kit 根目錄:", ROOT)
print("SVD 模型:", MODEL_ROOT, "| unet.xml 存在:", (MODEL_ROOT / "model" / "unet.xml").exists())


## 1. 載入 OpenVINO IR（第一次 compile 較慢，之後快）

四個組件：image_encoder（CLIP）、unet（去噪）、vae_encoder / vae_decoder。


In [ ]:
DEVICE = "CPU"   # 可試 "AUTO"
core = ov.Core()

model_dir = MODEL_ROOT / "model"
vae_encoder = core.compile_model(model_dir / "vae_encoder.xml", DEVICE)
image_encoder = core.compile_model(model_dir / "image_encoder.xml", DEVICE)
unet = core.compile_model(model_dir / "unet.xml", DEVICE)
vae_decoder = core.compile_model(model_dir / "vae_decoder.xml", DEVICE)

scheduler = AnimateLCMSVDStochasticIterativeScheduler.from_pretrained(MODEL_ROOT / "scheduler")
feature_extractor = CLIPImageProcessor.from_pretrained(MODEL_ROOT / "feature_extractor")

pipe = OVStableVideoDiffusionPipeline(
    vae_encoder, image_encoder, unet, vae_decoder, scheduler, feature_extractor
)
print("pipeline ready")


## 2. 生成短片

- `profile` 建議先用 `"low"`（14 幀 / 4 step）量一次實際時間，再用 `"high"`。
- 動作幅度：`MOTION`（motion_bucket_id，40–127，越大越動）。
- 時間預期（i5-13600K 級）：一支 2–4 秒片段數分鐘級（實測後可依結果取捨）。


In [ ]:
# ==== 設定 ====
IMAGE_PATH = str(ROOT / "input.png")   # ← 換成你的圖（建議 16:9）
PROFILE = "low"                        # "low" | "high"
MOTION = 80
SEED = 42
FPS = 7
WIDTH, HEIGHT = 512, 320

PRESETS = {
    "low":  {"num_frames": 14, "num_inference_steps": 4, "decode_chunk_size": 7},
    "high": {"num_frames": 25, "num_inference_steps": 6, "decode_chunk_size": 8},
}
preset = PRESETS[PROFILE]

image = load_image(IMAGE_PATH).resize((WIDTH, HEIGHT))
generator = torch.Generator(device="cpu").manual_seed(SEED)

t0 = time.perf_counter()
frames = pipe(
    image,
    width=WIDTH, height=HEIGHT,
    num_frames=preset["num_frames"],
    num_inference_steps=preset["num_inference_steps"],
    decode_chunk_size=preset["decode_chunk_size"],
    motion_bucket_id=MOTION,
    fps=FPS,
    generator=generator,
    output_type="pil",
).frames[0]
print(f"生成完成：{len(frames)} 幀，{time.perf_counter()-t0:.0f} 秒")


## 3. 存成 mp4（用 kit 內的 ffmpeg 編碼，不需額外套件）

In [ ]:
def find_ffmpeg():
    import os
    env = os.environ.get("FFMPEG")
    if env and Path(env).exists():
        return env
    for cand in (ROOT / "ffmpeg" / "bin" / "ffmpeg.exe", ROOT / "ffmpeg" / "bin" / "ffmpeg"):
        if cand.exists():
            return str(cand)
    found = shutil.which("ffmpeg")
    if found:
        return found
    raise RuntimeError("找不到 ffmpeg（設 FFMPEG 環境變數或放到 ffmpeg/bin/）")

out_path = ROOT / "work" / "svd_clip.mp4"
out_path.parent.mkdir(parents=True, exist_ok=True)

tmp = Path(tempfile.mkdtemp(prefix="svd_frames_"))
for i, f in enumerate(frames):
    f.save(tmp / f"{i:04d}.png")
subprocess.run([
    find_ffmpeg(), "-y", "-loglevel", "error",
    "-framerate", str(FPS), "-i", str(tmp / "%04d.png"),
    "-c:v", "libx264", "-pix_fmt", "yuv420p", "-crf", "18",
    str(out_path),
], check=True)
shutil.rmtree(tmp, ignore_errors=True)
print("輸出:", out_path)

try:
    from IPython.display import Video, display
    display(Video(str(out_path), embed=False))
except Exception:
    pass


## 4. 之後怎麼用

- 把 `work/svd_clip.mp4` 當片頭/過場，跟主影片（`work/final.mp4`）用 ffmpeg 拼接：
  ```
  ffmpeg -i final.mp4 -i svd_clip.mp4 -filter_complex "[0:v][0:a][1:v][1:a]concat=n=2:v=1:a=1" merged.mp4
  ```
- 換不同的 MOTION / SEED 多生幾段挑喜歡的。
- 批量化：改用命令列 `python scripts/07_svd_video.py in.png out.mp4 --profile low --motion 100`。
